In [3]:
!pip -q install sentence-transformers faiss-cpu transformers accelerate sentencepiece

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------
# Knowledge Base
# -------------------------
documents = [
    "The Eiffel Tower is located in Paris, France and was completed in 1889.",
    "Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.",
    "Python is a popular high-level programming language used in AI development.",
    "Vector databases store embeddings and support fast similarity search."
]

# -------------------------
# Embedding Model
# -------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

doc_embeddings = embed_model.encode(documents).astype("float32")

# -------------------------
# FAISS Index
# -------------------------
dimension = doc_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

# -------------------------
# Query
# -------------------------
query = "What is RAG in AI?"

query_embedding = embed_model.encode([query]).astype("float32")

D, I = index.search(query_embedding, k=2)

retrieved_chunks = [documents[i] for i in I[0]]

# -------------------------
# Prompt
# -------------------------
context = " ".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
{query}

Answer:
"""

# -------------------------
# Load FLAN-T5
# -------------------------
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=60
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# -------------------------
# Output
# -------------------------
print("="*60)
print("QUESTION")
print("="*60)
print(query)

print("\n")

print("="*60)
print("RETRIEVED DOCUMENTS")
print("="*60)

for i, doc in enumerate(retrieved_chunks, 1):
    print(f"{i}. {doc}")

print("\n")

print("="*60)
print("GENERATED ANSWER")
print("="*60)

print(generated_text)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

QUESTION
What is RAG in AI?


RETRIEVED DOCUMENTS
1. Retrieval-Augmented Generation (RAG) combines document retrieval with text generation.
2. Python is a popular high-level programming language used in AI development.


GENERATED ANSWER
combines document retrieval with text generation
